In [1]:
import pandas as pd

df=pd.read_csv("../data/raw/Metro_zori_uc_sfrcondomfr_sm_month.csv")
df.head()


,RegionID,SizeRank,RegionName,RegionType,StateName,2015-01-31,2015-02-28,2015-03-31,2015-04-30,2015-05-31,...,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31
0,102001,0,United States,country,NaN,1150.458020,1156.813729,1165.483604,1174.328410,1183.035007,...,1917.015885,1913.158141,1909.613807,1911.099804,1918.090366,1928.717336,1939.279576,1948.758837,1956.535779,1962.188014
1,394913,1,"New York, NY",msa,NY,2283.102284,2298.430193,2318.004173,2337.299847,2352.535819,...,3480.248872,3459.329392,3447.633327,3445.764531,3464.250818,3494.974421,3527.038595,3565.477382,3597.970728,3627.457796
2,753899,2,"Los Angeles, CA",msa,CA,1754.647191,1765.832563,1781.036708,1796.290752,1811.474759,...,2905.510423,2900.366545,2892.220790,2894.669544,2901.080983,2914.634187,2924.958510,2934.850614,2941.837391,2944.420416
3,394463,3,"Chicago, IL",msa,IL,1380.831174,1388.440562,1398.710597,1408.491034,1418.255705,...,2141.436367,2138.222886,2136.829818,2148.041168,2166.006337,2188.262661,2208.104728,2227.663791,2244.133827,2252.637940
4,394514,4,"Dallas, TX",msa,TX,1056.351903,1061.180439,1068.880404,1080.006784,1089.134956,...,1653.561913,1648.184400,1643.125933,1640.774793,1643.861562,1650.722236,1659.693619,1664.270817,1667.030802,1667.042802


In [2]:
print("Shap",df.shape)
print("Columns Name",df.columns)
print("---------------------------")
print("Info",df.info())

Shap (752, 144)
Columns Name Index(['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName',
       '2015-01-31', '2015-02-28', '2015-03-31', '2015-04-30', '2015-05-31',
       ...
       '2025-10-31', '2025-11-30', '2025-12-31', '2026-01-31', '2026-02-28',
       '2026-03-31', '2026-04-30', '2026-05-31', '2026-06-30', '2026-07-31'],
      dtype='object', length=144)
---------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 752 entries, 0 to 751
Columns: 144 entries, RegionID to 2026-07-31
dtypes: float64(139), int64(2), object(3)
memory usage: 846.1+ KB
Info None


# new concept

In [3]:
# melt function Keep these identifier columns fixed,
# and turn all the other columns into rows.
id_columns = [
    "RegionID",
    "SizeRank",
    "RegionName",
    "RegionType",
    "StateName"
]
df_long=df.melt(
    id_vars=id_columns,
    var_name="date",
    value_name="rent"
)


In [4]:
df_long["date"] = pd.to_datetime(df_long["date"])

In [5]:
df_long.head(10)

,RegionID,SizeRank,RegionName,RegionType,StateName,date,rent
0,102001,0,United States,country,NaN,2015-01-31,1150.458020
1,394913,1,"New York, NY",msa,NY,2015-01-31,2283.102284
2,753899,2,"Los Angeles, CA",msa,CA,2015-01-31,1754.647191
3,394463,3,"Chicago, IL",msa,IL,2015-01-31,1380.831174
4,394514,4,"Dallas, TX",msa,TX,2015-01-31,1056.351903
5,394692,5,"Houston, TX",msa,TX,2015-01-31,1209.965584
6,395209,6,"Washington, DC",msa,VA,2015-01-31,1719.530882
7,394974,7,"Philadelphia, PA",msa,PA,2015-01-31,1199.439512
8,394856,8,"Miami, FL",msa,FL,2015-01-31,1467.919657
9,394347,9,"Atlanta, GA",msa,GA,2015-01-31,997.598613


In [6]:
print(df_long.shape)
df_long.info()

(104528, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104528 entries, 0 to 104527
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   RegionID    104528 non-null  int64         
 1   SizeRank    104528 non-null  int64         
 2   RegionName  104528 non-null  object        
 3   RegionType  104528 non-null  object        
 4   StateName   104389 non-null  object        
 5   date        104528 non-null  datetime64[ns]
 6   rent        52768 non-null   float64       
dtypes: datetime64[ns](1), float64(1), int64(2), object(3)
memory usage: 5.6+ MB


In [7]:
df_long.isnull().sum()

RegionID          0
SizeRank          0
RegionName        0
RegionType        0
StateName       139
date              0
rent          51760
dtype: int64

In [8]:
df_long[df_long["StateName"].isnull()][["RegionName", "RegionType", "StateName"]].drop_duplicates()

,RegionName,RegionType,StateName
0,United States,country,NaN


In [9]:
df_long = df_long[df_long["RegionType"] == "msa"].copy()

In [10]:
print("Rows:", len(df_long))
print("Missing rent:", df_long["rent"].isnull().sum())
print("Missing %:",round(df_long["rent"].isnull().mean() * 100, 2))

Rows: 104389
Missing rent: 51760
Missing %: 49.58


In [11]:
missing_by_date = (df_long.groupby("date")["rent"].apply(lambda x: x.isnull().mean() * 100).round(2))

missing_by_date.head(20)

date
2015-01-31    71.90
2015-02-28    71.77
2015-03-31    71.24
2015-04-30    70.57
2015-05-31    70.57
2015-06-30    70.31
2015-07-31    70.44
2015-08-31    70.31
2015-09-30    70.17
2015-10-31    70.04
2015-11-30    69.91
2015-12-31    69.11
2016-01-31    68.71
2016-02-29    68.71
2016-03-31    68.31
2016-04-30    67.91
2016-05-31    66.05
2016-06-30    65.78
2016-07-31    65.51
2016-08-31    65.25
Name: rent, dtype: float64

In [12]:
missing_by_date.tail(20)

date
2024-12-31    25.57
2025-01-31    23.44
2025-02-28    22.64
2025-03-31    21.57
2025-04-30    21.70
2025-05-31    20.64
2025-06-30    20.37
2025-07-31    20.24
2025-08-31    19.71
2025-09-30    19.44
2025-10-31    18.51
2025-11-30    18.11
2025-12-31    16.91
2026-01-31    15.18
2026-02-28    14.25
2026-03-31    11.58
2026-04-30    10.12
2026-05-31     7.86
2026-06-30     6.39
2026-07-31     0.27
Name: rent, dtype: float64

In [13]:
missing_by_region = (
    df_long
    .groupby(["RegionID", "RegionName"])["rent"]
    .apply(lambda x: x.isnull().mean() * 100)
    .round(2)
    .sort_values(ascending=False)
)

missing_by_region.head(20)

RegionID  RegionName       
394781    Lebanon, MO          99.28
753903    Marietta, OH         99.28
394702    Hutchinson, MN       99.28
395208    Washington, NC       99.28
395228    Willmar, MN          99.28
394793    Liberal, KS          99.28
394739    Kendallville, IN     99.28
394625    Galesburg, IL        99.28
394584    Fallon, NV           99.28
394498    Cordele, GA          99.28
394915    Newport, TN          99.28
394973    Peru, IN             99.28
394335    Arcadia, FL          99.28
394316    Alice, TX            99.28
394324    Americus, GA         99.28
394508    Crescent City, CA    99.28
394963    Paris, TX            99.28
394986    Plattsburgh, NY      99.28
394864    Mineral Wells, TX    99.28
394844    McComb, MS           99.28
Name: rent, dtype: float64

In [14]:
df_long = df_long.dropna(subset=["rent"]).copy()

In [15]:
df_long.isnull().sum()

RegionID      0
SizeRank      0
RegionName    0
RegionType    0
StateName     0
date          0
rent          0
dtype: int64

In [16]:
df_long = df_long.drop(columns=["SizeRank","RegionType"])

In [17]:
df_long.shape

(52629, 5)

In [18]:
df_long.head()

,RegionID,RegionName,StateName,date,rent
1,394913,"New York, NY",NY,2015-01-31,2283.102284
2,753899,"Los Angeles, CA",CA,2015-01-31,1754.647191
3,394463,"Chicago, IL",IL,2015-01-31,1380.831174
4,394514,"Dallas, TX",TX,2015-01-31,1056.351903
5,394692,"Houston, TX",TX,2015-01-31,1209.965584


In [19]:
df_long=df_long.rename(columns={
    "RegionID": "zillow_region_id",
    "RegionName": "region_name",
    "StateName": "state_name"
})
df_long["rent"]=df_long["rent"].round(2)

In [20]:
df_long.head()

,zillow_region_id,region_name,state_name,date,rent
1,394913,"New York, NY",NY,2015-01-31,2283.10
2,753899,"Los Angeles, CA",CA,2015-01-31,1754.65
3,394463,"Chicago, IL",IL,2015-01-31,1380.83
4,394514,"Dallas, TX",TX,2015-01-31,1056.35
5,394692,"Houston, TX",TX,2015-01-31,1209.97


In [21]:
df_redfin=pd.read_csv("../data/processed/redfin_monthly_housing.csv")

In [50]:
zillow_geo = (df_long[["zillow_region_id", "region_name", "state_name"]].drop_duplicates().copy())
redfin_geo = (df_redfin[["redfin_region_id", "region_name"]].drop_duplicates().copy())


In [51]:
zillow_geo["match_name"] = (zillow_geo["region_name"].str.lower().str.strip())
redfin_geo["match_name"] = (redfin_geo["region_name"].str.lower().str.replace(" metro area", "", regex=False).str.strip())

In [52]:
zillow_geo.head()

,zillow_region_id,region_name,state_name,match_name
1,394913,"New York, NY",NY,"new york, ny"
2,753899,"Los Angeles, CA",CA,"los angeles, ca"
3,394463,"Chicago, IL",IL,"chicago, il"
4,394514,"Dallas, TX",TX,"dallas, tx"
5,394692,"Houston, TX",TX,"houston, tx"


In [25]:
zillow_geo=set(zillow_geo["match_name"].astype(str))
redfin_geo=set(redfin_geo["match_name"].astype(str))

In [26]:
print("Total Zillow metro",len(zillow_geo))


Total Zillow metro 751


In [30]:

matches= zillow_geo & redfin_geo
not_matching = zillow_geo - redfin_geo
print("Direct name matches",len(matches))
print("Still unmatched",len(not_matching))

Direct name matches 748
Still unmatched 3


In [31]:
print(not_matching)

{'ca±on city, co', 'urban honolulu, hi', 'winston, nc'}


In [44]:
redfin_geo_df = (
    df_redfin[
        ["redfin_region_id", "region_name"]
    ]
    .drop_duplicates()
    .copy()
)

redfin_geo_df["match_name"] = (
    redfin_geo_df["region_name"]
    .str.lower()
    .str.replace(" metro area", "", regex=False)
    .str.strip()
)

In [46]:
redfin_geo_df.head()

,redfin_region_id,region_name,match_name
0,10100,"Aberdeen, SD metro area","aberdeen, sd"
139,10140,"Aberdeen, WA metro area","aberdeen, wa"
278,10180,"Abilene, TX metro area","abilene, tx"
417,10220,"Ada, OK metro area","ada, ok"
555,10300,"Adrian, MI metro area","adrian, mi"


In [48]:
redfin_geo_df[
    redfin_geo_df["match_name"].str.contains(
        "honolulu|winston|canon|cañon",
        case=False,
        na=False,
        regex=True
    )
][["redfin_region_id", "region_name", "match_name"]]

,redfin_region_id,region_name,match_name
18958,15860,"Cañon City, CO metro area","cañon city, co"
119616,46520,"Honolulu, HI metro area","honolulu, hi"
127662,49180,"Winston-Salem, NC metro area","winston-salem, nc"


In [ ]:
name_fixes = {
    "ca±on city, co": "cañon city, co",
    "urban honolulu, hi": "honolulu, hi",
    "winston, nc": "winston-salem, nc"
}

zillow_geo["match_name"] = (zillow_geo["match_name"].replace(name_fixes))

In [54]:
zillow_name_set = set(zillow_geo["match_name"])
redfin_name_set = set(redfin_geo_df["match_name"])

matches = zillow_name_set & redfin_name_set
not_matching = zillow_name_set - redfin_name_set

print("Total Zillow metros:", len(zillow_name_set))
print("Direct matches:", len(matches))
print("Still unmatched:", len(not_matching))

print("Coverage:",round(len(matches) / len(zillow_name_set) * 100, 2),"%")

Total Zillow metros: 751
Direct matches: 751
Still unmatched: 0
Coverage: 100.0 %


In [55]:
zillow_redfin_bridge = zillow_geo.merge(redfin_geo_df[["redfin_region_id", "match_name"]],
    on="match_name",
    how="left",
    validate="one_to_one"
)

In [59]:

print(zillow_redfin_bridge["redfin_region_id"].isnull().sum())

0


In [60]:
df_long = df_long.merge(zillow_redfin_bridge[["zillow_region_id", "redfin_region_id"]],
    on="zillow_region_id",
    how="left",
    validate="many_to_one"
)

In [62]:
df_long.head()

,zillow_region_id,region_name,state_name,date,rent,redfin_region_id
0,394913,"New York, NY",NY,2015-01-31,2283.10,35614
1,753899,"Los Angeles, CA",CA,2015-01-31,1754.65,31084
2,394463,"Chicago, IL",IL,2015-01-31,1380.83,16984
3,394514,"Dallas, TX",TX,2015-01-31,1056.35,19124
4,394692,"Houston, TX",TX,2015-01-31,1209.97,26420


In [64]:
df_long = df_long[
    [
        "zillow_region_id",
        "redfin_region_id",
        "region_name",
        "state_name",
        "date",
        "rent"
    ]
].copy()

df_long["zillow_region_id"] = df_long["zillow_region_id"].astype(str)
df_long["redfin_region_id"] = df_long["redfin_region_id"].astype(str)

df_long["date"] = pd.to_datetime(df_long["date"])

df_long["rent"] = pd.to_numeric(
    df_long["rent"],
    errors="coerce"
)

df_long["rent"] = df_long["rent"].round(2)

df_long = (
    df_long
    .sort_values(["redfin_region_id", "date"])
    .reset_index(drop=True)
)

In [65]:
def validate_zillow_rent(df):

    assert df["zillow_region_id"].notna().all(), \
        "Missing Zillow region IDs detected"

    assert df["redfin_region_id"].notna().all(), \
        "Missing Redfin region IDs detected"

    assert df["region_name"].notna().all(), \
        "Missing region names detected"

    assert df["date"].notna().all(), \
        "Missing dates detected"

    assert df["rent"].notna().all(), \
        "Missing rent values detected"

    assert (df["rent"] > 0).all(), \
        "Invalid rent values detected"

    assert not df.duplicated(
        subset=["zillow_region_id", "date"]
    ).any(), \
        "Duplicate Zillow region-date records detected"

In [67]:
df_long.to_csv("data/processed/zillow_monthly_rent.csv",index=False)


OSError: Cannot save file into a non-existent directory: 'data\processed'